In [ ]:
import torch




In [ ]:
def knn_graph(
    x,
    k=10,
    chunk_size=512
):
    n = x.size(0)

    if n <= 1:
        raise ValueError(
            "at least two samples are required"
        )

    k = min(
        k,
        n - 1
    )

    x_norm = (
        x * x
    ).sum(
        dim=1
    )

    edges = []

    for start in range(
        0,
        n,
        chunk_size
    ):
        end = min(
            start + chunk_size,
            n
        )

        chunk = x[start:end]

        distance = (
            x_norm[start:end].unsqueeze(1)
            + x_norm.unsqueeze(0)
            - 2.0 * (
                chunk @ x.t()
            )
        )

        distance = torch.clamp(
            distance,
            min=0.0
        )

        _, indices = torch.topk(
            distance,
            k=k + 1,
            dim=1,
            largest=False
        )

        indices = indices[:, 1:]

        rows = torch.arange(
            start,
            end,
            device=x.device
        ).unsqueeze(1)

        rows = rows.expand_as(
            indices
        )

        edges.append(
            torch.stack(
                [
                    rows.reshape(-1),
                    indices.reshape(-1)
                ],
                dim=0
            )
        )

    edges = torch.cat(
        edges,
        dim=1
    )

    reverse_edges = edges.flip(
        0
    )

    edges = torch.cat(
        [
            edges,
            reverse_edges
        ],
        dim=1
    )

    values = torch.ones(
        edges.size(1),
        device=x.device,
        dtype=x.dtype
    )

    graph = torch.sparse_coo_tensor(
        edges,
        values,
        size=(n, n),
        device=x.device
    ).coalesce()

    graph = torch.sparse_coo_tensor(
        graph.indices(),
        torch.ones(
            graph.indices().size(1),
            device=x.device,
            dtype=x.dtype
        ),
        size=(n, n),
        device=x.device
    ).coalesce()

    return graph


In [ ]:
def graph_laplacian(graph):
    indices = graph.indices()
    values = graph.values()

    n = graph.size(0)

    degree = torch.zeros(
        n,
        device=graph.device,
        dtype=values.dtype
    )

    degree.scatter_add_(
        0,
        indices[0],
        values
    )

    return degree


In [ ]:
def manifold_regularization(
    z,
    graph,
    degree
):
    indices = graph.indices()
    values = graph.values()

    source = indices[0]
    target = indices[1]

    z_squared = (
        z * z
    ).sum(
        dim=1
    )

    degree_term = torch.sum(
        degree * z_squared
    )

    adjacency_term = torch.sum(
        values
        * (
            z[source]
            * z[target]
        ).sum(
            dim=1
        )
    )

    return degree_term - adjacency_term


In [ ]:
def manifold_matching_loss(
    transport,
    cost,
    z,
    graph,
    degree,
    lambda_graph=0.05
):
    transport_cost = torch.sum(
        transport * cost
    )

    graph_cost = manifold_regularization(
        z,
        graph,
        degree
    )

    loss = (
        transport_cost
        + lambda_graph * graph_cost
    )

    return (
        loss,
        transport_cost,
        graph_cost
    )